## ***NLP***
- ***NLP is a field of AI that enables machines to process, understand and generate human language using techniques from machine learning, deep learning and linguistics.***

### ***RNN***
- ***RNN remembers information from previous time steps(having vanishing gradient problem).***

### ***LSTM (resolve rnn problem)***
- ***LSTM was designed to solve the long-term dependency problem of vanilla RNNs & it's uses gates to decide what information to keep and updated.***

- ***LSTM is an advanced form of RNN that uses gated memory and a separate cell state to handle long-term dependencies more effectively than vanilla RNN.***

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("../content/imdb_data.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:
df.shape

(50000, 2)

In [7]:
df.isnull().sum()

,0
review,0
sentiment,0


In [8]:
df.drop_duplicates(inplace=True)

In [9]:
df.shape

(49582, 2)

- ***Text Pre-Processing***

In [10]:
## Converting to ;owercase
df["review"] = df["review"].str.lower()

# removing urls
import re
def remove_urls(text):
    text = re.sub(r"http\S+" , "", text)  # (pattern, repl, string) eg - https://www.google.com
    return text

df["review"] = df["review"].apply(remove_urls)

# removing punctuations
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "", text) # A-Z a-z 0-9 \s
    return text

df["review"] = df["review"].apply(remove_punctuations)

In [12]:
## removing HTML
def remove_html(text):
    text = re.sub(r"<.*?>" , "", text)
    return text

df["review"] = df["review"].apply(remove_html)

In [13]:
# remove stopwords
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [14]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def remove_stopwords(text):
  tokens = word_tokenize(text)
  stop_words = stopwords.words("english")

  for word in tokens:
    if word in stop_words:
      text = text.replace(word,"")
  return text
df["review"] = df["review"].apply(remove_stopwords)

In [15]:
## Stemming
# running -> run
# played -> play

from nltk.stem import PorterStemmer

def stemming(text):
  ps = PorterStemmer()
  stemmed_words = []
  tokens = word_tokenize(text)

  for token in tokens:
    stemmed_token = ps.stem(token)
    stemmed_words.append(stemmed_token)
  return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [16]:
df.sample(10)

,review,sentiment
49400,sw flm or dy snt fe flm festvl delght ts hones...,positive
23083,serious img spred tlent one film wh huge budge...,positive
26701,love exotc scenc fcnfntsi move th one unplesnt...,negative
21703,movi could terestg chrcter studi d could given...,negative
9629,predict parodi fail impress throughout loooooo...,negative
21641,fmou show mster enter elevr girlfriend suddenl...,negative
34449,know e s suppos comnt user comnt wll sy lot ne...,positive
24651,ccount seem rel humn fcr dded mx lot sdness m ...,positive
13015,clled remk good orgnl one cross border flyng c...,positive
15239,ctulli lke orgnl th flm ts up down here fewbr ...,positive


- ***Encoding***

In [17]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

In [18]:
y = df["sentiment"]

- ***Vectorization***

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)
X = tf.fit_transform(df["review"])

- ***Build Dataset & DataLoaders***

In [20]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.3,random_state=42
)

In [21]:
X_train.shape

(34707, 5000)

In [22]:
X_test.shape

(14875, 5000)

In [26]:

import torch
from torch.utils.data import TensorDataset, DataLoader

train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

#### ***Build Our RNN Model***

In [27]:
import torch.nn as nn
import torch.optim as optim

class RNN(nn.Module):
  def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

  def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [28]:
input_size = X_train.shape[1]
model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [32]:
# Train Our RNN

epochs = 50
for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction

        outputs = model(Xb) # (batch_size, 1)
        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch:{epoch+1}/{epochs} & loss:{loss.item()}")

epoch:1/50 & loss:0.1438092142343521
epoch:2/50 & loss:0.12024635821580887
epoch:3/50 & loss:0.22615912556648254
epoch:4/50 & loss:0.20234549045562744
epoch:5/50 & loss:0.04755118489265442
epoch:6/50 & loss:0.09062546491622925
epoch:7/50 & loss:0.44509461522102356
epoch:8/50 & loss:0.781308650970459
epoch:9/50 & loss:0.09894747287034988
epoch:10/50 & loss:0.17575721442699432
epoch:11/50 & loss:0.29329532384872437
epoch:12/50 & loss:0.20970050990581512
epoch:13/50 & loss:0.0820564553141594
epoch:14/50 & loss:0.10860217362642288
epoch:15/50 & loss:0.19874586164951324
epoch:16/50 & loss:0.19011326134204865
epoch:17/50 & loss:0.47719821333885193
epoch:18/50 & loss:0.15754666924476624
epoch:19/50 & loss:0.14541365206241608
epoch:20/50 & loss:0.17368265986442566
epoch:21/50 & loss:0.36541247367858887
epoch:22/50 & loss:0.16840362548828125
epoch:23/50 & loss:0.18590779602527618
epoch:24/50 & loss:0.1216631680727005
epoch:25/50 & loss:0.09389187395572662
epoch:26/50 & loss:0.17251752316951752


In [33]:
# evaluate metrics
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()
    print(f"accuracy:{correct_vals/tot_vals*100}")

accuracy:84.72605042016806


In [35]:
# save our model
saved_model = torch.save(model.state_dict(), "rnn_model.pt")
print(f"model saved successfully..✅")

model saved successfully..✅
